In [1]:
### thuisarts

import time
import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE_URL = "https://www.thuisarts.nl"
START_URL = "https://www.thuisarts.nl/overzicht/onderwerpen"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

CANCER_KEYWORDS = [
    "kanker", "carcinool", "onco", "tumor", "tumoren", "melanoom", 
    "sarcoom", "leukemie", "lymfoom", "cin2", "cin3", "bestraling", "chemo"
]

def get_cancer_topics():
    """Extracts cancer-related topic entry points from the main alphabetical directory."""
    print(f"Scanning Thuisarts master index: {START_URL}")
    response = requests.get(START_URL, headers=HEADERS)
    if response.status_code != 200:
        return {}

    soup = BeautifulSoup(response.text, 'html.parser')
    topic_links = {}
    content_area = soup.find('main') or soup.find('div', class_='content') or soup
    
    for link in content_area.find_all('a', href=True):
        href = link['href']
        name = link.get_text(strip=True)
        
        if any(kw in name.lower() or kw in href.lower() for kw in CANCER_KEYWORDS):
            full_url = BASE_URL + href if href.startswith('/') else href
            if name and "over ons" not in name.lower() and "zoeken" not in name.lower():
                if full_url not in topic_links.values():
                    topic_links[name] = full_url
    return topic_links

def format_html_list(element, indent_level=0):
    """Recursively formats HTML lists (ul, li) into clean indented text blocks."""
    items = []
    indent = "  " * indent_level
    bullet = "• " if indent_level == 0 else "◦ "
    
    for child in element.children:
        if child.name == 'li':
            # Extract only the immediate text, before nested lists
            text_parts = []
            for g_child in child.contents:
                if g_child.name not in ['ul', 'ol']:
                    if isinstance(g_child, str) or hasattr(g_child, 'get_text'):
                        t = g_child.get_text().strip() if hasattr(g_child, 'get_text') else str(g_child).strip()
                        if t: text_parts.append(t)
            
            immediate_text = " ".join(text_parts)
            if immediate_text:
                items.append(f"{indent}{bullet}{immediate_text}")
            
            # Recurse if there's a nested list inside this item
            nested_list = child.find(['ul', 'ol'])
            if nested_list:
                items.extend(format_html_list(nested_list, indent_level + 1))
                
        elif child.name in ['ul', 'ol']:
            items.extend(format_html_list(child, indent_level))
            
    return items

def extract_page_data(title, url, parent_title=None):
    """Hits an explicit web target and extracts both 'In het kort' and standard details."""
    try:
        response = requests.get(url, headers=HEADERS)
        if response.status_code != 200:
            return None
            
        soup = BeautifulSoup(response.text, 'html.parser')
        summary_text = "N/A"
        in_het_kort_container = soup.find('div', id='in-het-kort')
        
        if in_het_kort_container:
            body_item = in_het_kort_container.find('div', class_='field--name-body')
            if body_item:
                # Use our smart list formatter to handle inner/outer bullet points perfectly
                summary_lines = format_html_list(body_item)
                if summary_lines:
                    summary_text = "\n".join(summary_lines)
            
            if summary_text == "N/A":
                summary_text = in_het_kort_container.get_text(" ", strip=True)

        # Extract remaining paragraphs
        main_content = soup.find('main') or soup.find('article') or soup
        paragraphs = []
        for p in main_content.find_all(['p', 'li']):
            if in_het_kort_container and p.find_parent('div', id='in-het-kort'):
                continue
            txt = p.get_text(" ", strip=True)
            if txt and len(txt) > 20 and not any(s in txt.lower() for s in ["lees voor", "download als pdf", "deel deze pagina"]):
                paragraphs.append(txt)
                
        full_body = " ".join(paragraphs[:15])
        
        return {
            "Topic Category": parent_title if parent_title else title,
            "Page Title": title,
            "URL": url,
            "In Brief Summary": summary_text,
            "Content Extract": full_body if full_body else "N/A"
        }
    except Exception as e:
        print(f"Error parsing page {url}: {e}")
        return None

def deep_scrape_topic(topic_name, topic_url):
    """Scrapes the landing page AND searches for nested subpages to extract everything."""
    print(f"Evaluating Landing Hub: [{topic_name}] -> {topic_url}")
    results = []
    
    # FIX: Always scrape the parent hub page first to get its unique summary!
    parent_data = extract_page_data(topic_name, topic_url)
    if parent_data:
        results.append(parent_data)
    
    try:
        response = requests.get(topic_url, headers=HEADERS)
        if response.status_code != 200:
            return results
            
        soup = BeautifulSoup(response.text, 'html.parser')
        main_area = soup.find('main') or soup
        sub_links = {}
        
        # Look for sub-articles split under the topic base path
        for link in main_area.find_all('a', href=True):
            href = link['href']
            sub_title = link.get_text(strip=True)
            topic_slug = topic_url.replace(BASE_URL, '')
            
            if href.startswith(topic_slug) and len(href.split('/')) > len(topic_slug.split('/')):
                full_sub_url = BASE_URL + href if href.startswith('/') else href
                if full_sub_url != topic_url:
                    sub_links[sub_title] = full_sub_url

        if sub_links:
            print(f"--> Found {len(sub_links)} subpages under [{topic_name}]. Crawling branches...")
            for s_title, s_url in sub_links.items():
                p_data = extract_page_data(s_title, s_url, parent_title=topic_name)
                if p_data:
                    results.append(p_data)
                time.sleep(1.0)
                
        return results
    except Exception as e:
        print(f"Failed handling sublinks for {topic_name}: {e}")
        return results

def main():
    target_directory = get_cancer_topics()
    if not target_directory:
        print("Scrape cancelled: Mappings empty.")
        return

    master_data_pool = []
    for name, url in target_directory.items():
        records = deep_scrape_topic(name, url)
        master_data_pool.extend(records)
        time.sleep(1.5)

    if master_data_pool:
        df = pd.DataFrame(master_data_pool)
        df.to_csv("thuisarts_comprehensive_cancer_data.csv", index=False, encoding='utf-8-sig')
        df.to_excel("thuisarts_comprehensive_cancer_data.xlsx", index=False)
        print(f"\nSuccess! Exported {len(master_data_pool)} rows perfectly.")

if __name__ == "__main__":
    main()

Scanning Thuisarts master index: https://www.thuisarts.nl/overzicht/onderwerpen
Evaluating Landing Hub: [Alvleesklier-kanker] -> https://www.thuisarts.nl/alvleesklier-kanker
--> Found 4 subpages under [Alvleesklier-kanker]. Crawling branches...
Evaluating Landing Hub: [Asbestkanker in het longvlies of buikvlies] -> https://www.thuisarts.nl/asbestkanker-in-longvlies-of-buikvlies
--> Found 3 subpages under [Asbestkanker in het longvlies of buikvlies]. Crawling branches...
Evaluating Landing Hub: [Baarmoederhalskanker] -> https://www.thuisarts.nl/baarmoederhalskanker-0
--> Found 5 subpages under [Baarmoederhalskanker]. Crawling branches...
Evaluating Landing Hub: [Baarmoederkanker] -> https://www.thuisarts.nl/baarmoederkanker
--> Found 4 subpages under [Baarmoederkanker]. Crawling branches...
Evaluating Landing Hub: [Basaalcel-kanker] -> https://www.thuisarts.nl/basaalcel-kanker
--> Found 1 subpages under [Basaalcel-kanker]. Crawling branches...
Evaluating Landing Hub: [Blaaskanker] -> ht